In [1]:
# load libraries 
import pandas as pd
import numpy as np
import re
import math
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

In [5]:
# get the SUN covid study dataset
Covid51countries = pd.read_csv(Path("~/Projects/hypocognition/data/raw/Covid51countries.csv").expanduser())
Covid51countries

,country,countryname,StartDate,EndDate,Status,Progress,Duration,Finished,RecordedDate,home,...,gender,ladder,employment,losejob,covidsymptom,language,datasource,ISO3,longstring,na_count
0,10,Australia,04/05/2020 19:34,04/05/2020 19:40,0.0,100.0,337.0,1.0,04/05/2020 19:40,8.0,...,1.0,5.0,4,NaN,0.0,ZH-S,snowball,AUS,3,0
1,10,Australia,04/05/2020 20:26,04/05/2020 20:35,0.0,100.0,530.0,1.0,04/05/2020 20:35,9.0,...,2.0,7.0,3,NaN,NaN,ZH-S,snowball,AUS,3,0
2,10,Australia,05/05/2020 21:54,05/05/2020 22:07,0.0,100.0,793.0,1.0,05/05/2020 22:07,29.0,...,1.0,4.0,1,NaN,0.0,ZH-S,snowball,AUS,5,0
3,10,Australia,04/05/2020 20:29,04/05/2020 20:34,0.0,100.0,285.0,1.0,04/05/2020 20:34,73.0,...,2.0,5.0,4,NaN,0.0,ZH-S,snowball,AUS,4,0
4,10,Australia,19/04/2020 20:21,19/04/2020 20:27,0.0,100.0,367.0,1.0,19/04/2020 20:27,100.0,...,1.0,5.0,1,NaN,0.0,ZH-S,snowball,AUS,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24216,92,Kenya,25/04/2020 01:25,25/04/2020 02:44,0.0,100.0,4690.0,1.0,25/04/2020 02:44,100.0,...,2.0,4.0,4,NaN,0.0,EN,snowball,KEN,4,0
24217,92,Kenya,24/04/2020 11:06,24/04/2020 11:15,0.0,100.0,556.0,1.0,24/04/2020 11:15,100.0,...,1.0,6.0,5,0.0,0.0,EN,snowball,KEN,2,0
24218,92,Kenya,26/04/2020 10:16,26/04/2020 10:36,0.0,100.0,1218.0,1.0,26/04/2020 10:36,80.0,...,1.0,3.0,4,NaN,0.0,EN,snowball,KEN,3,0
24219,92,Kenya,23/04/2020 03:46,23/04/2020 04:13,0.0,100.0,1641.0,1.0,23/04/2020 04:13,3.0,...,1.0,3.0,4,NaN,0.0,EN,snowball,KEN,3,0


In [3]:
Covid51countries.columns

Index(['country', 'countryname', 'StartDate', 'EndDate', 'Status', 'Progress',
       'Duration', 'Finished', 'RecordedDate', 'home', 'gathering', 'distance',
       'hands', 'help_covid', 'donate_covid', 'volunteer_covid', 'help',
       'donate', 'volunteer', 'admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness',
       'ERQ1_rumination', 'ERQ2_reappraisal', 'ERQ3_suppression',
       'ERQ4_socialsharing', 'ERQ5_distraction', 'ERQ6_acceptance', 'support',
       'connected', 'phy_healthy', 'mentally_healthy', 'stressed', 'tired',
       'depressed', 'res_1', 'res_2', 'euda_1', 'euda_2', 'swl', 'sympathy',
       'concerned', 'overwhelmed', 'distressed', 'self_vulnerable',
       'country_vulnerable', 'age', 'education', 'gender', 'ladder',
       'employment', 'losejob', 'covidsymptom', 'languag

In [4]:
Covid51countries = Covid51countries[['country', 'countryname', 'language','ISO3', 'admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness']]

In [6]:
# add language names to the df
lang_names = pd.read_csv(Path("~/Projects/hypocognition/data/external/lang_names.csv"))
Covid51countries = Covid51countries.merge(lang_names, left_on="language", right_on="Code", how="left")
Covid51countries

,country,countryname,StartDate,EndDate,Status,Progress,Duration,Finished,RecordedDate,home,...,employment,losejob,covidsymptom,language,datasource,ISO3,longstring,na_count,Code,Language_Name
0,10,Australia,04/05/2020 19:34,04/05/2020 19:40,0.0,100.0,337.0,1.0,04/05/2020 19:40,8.0,...,4,NaN,0.0,ZH-S,snowball,AUS,3,0,ZH-S,Simplified Chinese
1,10,Australia,04/05/2020 20:26,04/05/2020 20:35,0.0,100.0,530.0,1.0,04/05/2020 20:35,9.0,...,3,NaN,NaN,ZH-S,snowball,AUS,3,0,ZH-S,Simplified Chinese
2,10,Australia,05/05/2020 21:54,05/05/2020 22:07,0.0,100.0,793.0,1.0,05/05/2020 22:07,29.0,...,1,NaN,0.0,ZH-S,snowball,AUS,5,0,ZH-S,Simplified Chinese
3,10,Australia,04/05/2020 20:29,04/05/2020 20:34,0.0,100.0,285.0,1.0,04/05/2020 20:34,73.0,...,4,NaN,0.0,ZH-S,snowball,AUS,4,0,ZH-S,Simplified Chinese
4,10,Australia,19/04/2020 20:21,19/04/2020 20:27,0.0,100.0,367.0,1.0,19/04/2020 20:27,100.0,...,1,NaN,0.0,ZH-S,snowball,AUS,3,0,ZH-S,Simplified Chinese
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24216,92,Kenya,25/04/2020 01:25,25/04/2020 02:44,0.0,100.0,4690.0,1.0,25/04/2020 02:44,100.0,...,4,NaN,0.0,EN,snowball,KEN,4,0,EN,English
24217,92,Kenya,24/04/2020 11:06,24/04/2020 11:15,0.0,100.0,556.0,1.0,24/04/2020 11:15,100.0,...,5,0.0,0.0,EN,snowball,KEN,2,0,EN,English
24218,92,Kenya,26/04/2020 10:16,26/04/2020 10:36,0.0,100.0,1218.0,1.0,26/04/2020 10:36,80.0,...,4,NaN,0.0,EN,snowball,KEN,3,0,EN,English
24219,92,Kenya,23/04/2020 03:46,23/04/2020 04:13,0.0,100.0,1641.0,1.0,23/04/2020 04:13,3.0,...,4,NaN,0.0,EN,snowball,KEN,3,0,EN,English


In [7]:
# merge croatian, bosnian and serbian
scb = ["Serbian", "Croatian", "Bosnian"]
Covid51countries = Covid51countries.drop(columns=['language'])
Covid51countries.loc[Covid51countries["Language_Name"].isin(scb), "Language_Name"] = "Serbian-Croatian-Bosnian"


In [8]:
# We want to use on only one language of responses for each country
# Further, that language cannot be English

# Calculate the number of responses for each country
country_response_count = pd.DataFrame(Covid51countries[['countryname', 'ISO3']].value_counts())

#collect the languages which people used to respond from each country
languages_per_country = Covid51countries.groupby('countryname')['Language_Name'].value_counts().reset_index(name="count")

# mereg num resposes to languages people used
languages_per_country = languages_per_country.merge(
    country_response_count,
    on="countryname",
    how="left"
)

# Clean up
languages_per_country = languages_per_country.rename(columns={"count_x": "lang_responses", "count_y": "total_responses"})
languages_per_country["percent"] = languages_per_country["lang_responses"] / languages_per_country["total_responses"] * 100

languages_per_country

,countryname,Language_Name,lang_responses,total_responses,percent
0,Australia,Simplified Chinese,181,378,47.883598
1,Australia,English,156,378,41.269841
2,Australia,Indonesian,12,378,3.174603
3,Australia,Traditional Chinese,10,378,2.645503
4,Australia,Vietnamese,5,378,1.322751
...,...,...,...,...,...
491,Vietnam,Simplified Chinese,2,338,0.591716
492,Vietnam,French,1,338,0.295858
493,Vietnam,Japanese,1,338,0.295858
494,Vietnam,Polish,1,338,0.295858


In [130]:
languages_per_country.to_csv(Path("~/Projects/hypocognition/data/processed/languages_per_country.csv").expanduser())

In [9]:
non_eng = languages_per_country[languages_per_country['Language_Name'] != "English"]
non_eng

,countryname,Language_Name,lang_responses,total_responses,percent
0,Australia,Simplified Chinese,181,378,47.883598
2,Australia,Indonesian,12,378,3.174603
3,Australia,Traditional Chinese,10,378,2.645503
4,Australia,Vietnamese,5,378,1.322751
5,Australia,Danish,3,378,0.793651
...,...,...,...,...,...
491,Vietnam,Simplified Chinese,2,338,0.591716
492,Vietnam,French,1,338,0.295858
493,Vietnam,Japanese,1,338,0.295858
494,Vietnam,Polish,1,338,0.295858


In [132]:
non_eng[non_eng['Language_Name']=="Dari"]

,countryname,Language_Name,lang_responses,total_responses,percent
308,Netherlands,Dari,3,1348,0.222552
425,Ukraine,Dari,1,698,0.143266
454,United Kingdom (UK),Dari,1,662,0.151057


In [133]:
# gather list of top names
try:
    del top4
except:
    pass
for n in non_eng['countryname'].unique():
    df = non_eng[non_eng['countryname'] == n].reset_index(drop=True)
    try:
        top4 = pd.concat([top4, df.iloc[0:len(df)]], ignore_index=True)
    except:
        top4 = df.iloc[0:len(df)]

top4

,countryname,Language_Name,lang_responses,total_responses,percent
0,Australia,Simplified Chinese,181,378,47.883598
1,Australia,Indonesian,12,378,3.174603
2,Australia,Traditional Chinese,10,378,2.645503
3,Australia,Vietnamese,5,378,1.322751
4,Australia,Danish,3,378,0.793651
...,...,...,...,...,...
441,Vietnam,Simplified Chinese,2,338,0.591716
442,Vietnam,French,1,338,0.295858
443,Vietnam,Japanese,1,338,0.295858
444,Vietnam,Polish,1,338,0.295858


In [134]:
# we are short the 2*50 rows we'd excpect.
top4['countryname'].value_counts()

countryname
Netherlands                       34
United Kingdom (UK)               31
United States of America (USA)    28
Germany                           26
France                            25
Spain                             21
Sweden                            20
Canada                            18
Hungary                           18
Italy                             17
Malta                             15
Australia                         14
Denmark                           13
Greece                            12
Japan                             11
Finland                            9
Singapore                          9
Brazil                             8
Israel                             8
New Zealand                        7
Malaysia                           7
Vietnam                            7
Turkey                             6
South Africa                       6
Ireland                            6
Mongolia                           6
Taiwan                    

In [135]:

covid_to_bila_nouns_full_lang_name_mapping = pd.read_csv(Path("~/Projects/hypocognition/data/external/covid_to_full_bila_lang_name_mapping.csv"))
covid_to_bila_nouns_full_lang_name_mapping

,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,EN,English,NaN,"Midland American English, Singlish, Devon, Sus..."
2,ID,Indonesian,Indonesian,"Standard Malay, Central Malay, Baba Malay"
3,ZH-T,Traditional Chinese,Mandarin Chinese,"Yue Chinese, Min Dong Chinese, Min Nan Chinese..."
4,VI,Vietnamese,Vietnamese,Nung (Viet Nam)
5,DA,Danish,Danish,NaN
6,AR,Arabic,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar..."
7,ES,Spanish,Latin American Spanish,"Spanish, Mexican Spanish"
8,AFRI,Afrikaans,Afrikaans,Dutch
9,ES-ES,Spanish (Spain),Spanish,NaN


In [136]:
covid_to_bila_nouns_full_lang_name_mapping['study_language_names'].nunique()

48

In [137]:
# Add the mapping to our selection table
selection = top4.merge(
    covid_to_bila_nouns_full_lang_name_mapping,
    left_on="Language_Name",
    right_on = "study_language_names",
    how = "left"
).drop(columns=["code", "study_language_names"])
selection


,countryname,Language_Name,lang_responses,total_responses,percent,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,Australia,Simplified Chinese,181,378,47.883598,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,Australia,Indonesian,12,378,3.174603,Indonesian,"Standard Malay, Central Malay, Baba Malay"
2,Australia,Traditional Chinese,10,378,2.645503,Mandarin Chinese,"Yue Chinese, Min Dong Chinese, Min Nan Chinese..."
3,Australia,Vietnamese,5,378,1.322751,Vietnamese,Nung (Viet Nam)
4,Australia,Danish,3,378,0.793651,Danish,NaN
...,...,...,...,...,...,...,...
441,Vietnam,Simplified Chinese,2,338,0.591716,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
442,Vietnam,French,1,338,0.295858,French,"Old French, Cajun French, Louisiana Creole French"
443,Vietnam,Japanese,1,338,0.295858,Japanese,Middle Chinese
444,Vietnam,Polish,1,338,0.295858,Polish,NaN


In [138]:
selection['Selected'] = np.nan
selection

,countryname,Language_Name,lang_responses,total_responses,percent,bila_language_name_mapping,possible_alternative_bila_language_name_mappings,Selected
0,Australia,Simplified Chinese,181,378,47.883598,Mandarin Chinese,"Beijing Mandarin, Wu Chinese",NaN
1,Australia,Indonesian,12,378,3.174603,Indonesian,"Standard Malay, Central Malay, Baba Malay",NaN
2,Australia,Traditional Chinese,10,378,2.645503,Mandarin Chinese,"Yue Chinese, Min Dong Chinese, Min Nan Chinese...",NaN
3,Australia,Vietnamese,5,378,1.322751,Vietnamese,Nung (Viet Nam),NaN
4,Australia,Danish,3,378,0.793651,Danish,NaN,NaN
...,...,...,...,...,...,...,...,...
441,Vietnam,Simplified Chinese,2,338,0.591716,Mandarin Chinese,"Beijing Mandarin, Wu Chinese",NaN
442,Vietnam,French,1,338,0.295858,French,"Old French, Cajun French, Louisiana Creole French",NaN
443,Vietnam,Japanese,1,338,0.295858,Japanese,Middle Chinese,NaN
444,Vietnam,Polish,1,338,0.295858,Polish,NaN,NaN


In [ ]:
# simplest way to get the table right is to just tweak it in excel and load it back in. Mark the "Selection" column of outline_selection.csv with 1 for the languages we want to use
selection.to_csv(Path("~/Projects/hypocognition/data/processed/outline_selection.csv").expanduser())


In [10]:
#Rename outline_selection.csv to selection_in_bila.csv and load it back in
selection = pd.read_csv(Path("~/Projects/hypocognition/data/processed/selection_in_bila.csv").expanduser(), index_col=0).reset_index(drop=True)
selection = selection[selection['Selected']==1].rename(columns={'study_language_names':"Language_Name"} )

In [11]:
# those are all good exclusions.
# Lets try filtering the dataset using our new selection
filtered = Covid51countries.merge(
    selection,
    on=['countryname', 'Language_Name'],
    how='inner'
)

filtered

,country,countryname,StartDate,EndDate,Status,Progress,Duration,Finished,RecordedDate,home,...,longstring,na_count,Code,Language_Name,lang_responses,total_responses,percent,bila_language_name_mapping,possible_alternative_bila_language_name_mappings,Selected
0,128,Netherlands,19/04/2020 04:38,19/04/2020 04:46,0.0,100.0,490.0,1.0,19/04/2020 04:46,99.0,...,2,0,AR,Arabic,144,1348,10.682493,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar...",1.0
1,128,Netherlands,18/04/2020 15:28,18/04/2020 15:45,0.0,100.0,1028.0,1.0,18/04/2020 15:45,95.0,...,1,0,AR,Arabic,144,1348,10.682493,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar...",1.0
2,128,Netherlands,19/04/2020 02:45,19/04/2020 03:06,0.0,100.0,1261.0,1.0,19/04/2020 03:06,29.0,...,2,0,AR,Arabic,144,1348,10.682493,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar...",1.0
3,128,Netherlands,19/04/2020 02:50,19/04/2020 03:01,0.0,100.0,669.0,1.0,19/04/2020 03:01,22.0,...,2,0,AR,Arabic,144,1348,10.682493,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar...",1.0
4,128,Netherlands,18/04/2020 11:18,18/04/2020 11:27,0.0,100.0,548.0,1.0,18/04/2020 11:27,95.0,...,6,0,AR,Arabic,144,1348,10.682493,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar...",1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,68,Germany,11/05/2020 19:29,11/05/2020 19:42,NaN,NaN,NaN,NaN,NaN,20.0,...,5,0,AR,Arabic,123,567,21.693122,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar...",1.0
340,68,Germany,11/05/2020 21:20,11/05/2020 21:33,NaN,NaN,NaN,NaN,NaN,80.0,...,3,0,AR,Arabic,123,567,21.693122,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar...",1.0
341,68,Germany,18/04/2020 14:28,18/04/2020 14:39,0.0,100.0,676.0,1.0,18/04/2020 14:39,46.0,...,5,0,AR,Arabic,123,567,21.693122,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar...",1.0
342,68,Germany,12/05/2020 09:18,12/05/2020 09:35,NaN,NaN,NaN,NaN,NaN,90.0,...,4,0,AR,Arabic,123,567,21.693122,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar...",1.0


In [13]:
filtered.columns

Index(['country', 'countryname', 'StartDate', 'EndDate', 'Status', 'Progress',
       'Duration', 'Finished', 'RecordedDate', 'home', 'gathering', 'distance',
       'hands', 'help_covid', 'donate_covid', 'volunteer_covid', 'help',
       'donate', 'volunteer', 'admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness',
       'ERQ1_rumination', 'ERQ2_reappraisal', 'ERQ3_suppression',
       'ERQ4_socialsharing', 'ERQ5_distraction', 'ERQ6_acceptance', 'support',
       'connected', 'phy_healthy', 'mentally_healthy', 'stressed', 'tired',
       'depressed', 'res_1', 'res_2', 'euda_1', 'euda_2', 'swl', 'sympathy',
       'concerned', 'overwhelmed', 'distressed', 'self_vulnerable',
       'country_vulnerable', 'age', 'education', 'gender', 'ladder',
       'employment', 'losejob', 'covidsymptom', 'datasou

In [142]:
# Check to make sure it worked
filtered.groupby('countryname')['Language_Name'].unique()

countryname
Germany        [Arabic]
Netherlands    [Arabic]
Turkey         [Arabic]
Name: Language_Name, dtype: object

In [143]:
filtered.groupby('countryname')['bila_language_name_mapping'].unique()

countryname
Germany        [Arabic]
Netherlands    [Arabic]
Turkey         [Arabic]
Name: bila_language_name_mapping, dtype: object

Goals
* For each distinct language, what is the lexical ellaboration for each of the 20 emotions?
* For each distinct language, is there any statistical significance in the distribution of the results of any particular emotion?

In [144]:
emotions = ['admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness']

In [145]:
# get the dataset of the dictionaries. We will look at how elaborated each of the twenty emotions are in each of the 39 languages
bila_nouns_full = pd.read_csv(Path("~/Projects/hypocognition/data/raw/bila_long_noun_lemmatized_full.csv").expanduser(), index_col=0)
bila_nouns_full

/tmp/ipykernel_1219494/268126284.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  bila_nouns_full = pd.read_csv(Path("~/Projects/hypocognition/data/raw/bila_long_noun_lemmatized_full.csv").expanduser(), index_col=0)


,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,ability,2.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-1.424169,10499,0.000190,1.098612
1,chi.14718491,accomplice,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.156552,10499,0.000000,0.000000
2,chi.14718491,account,14.0,9.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-7.520776,-2.250524,10499,0.000857,2.302585
3,chi.14718491,acre,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.444142,10499,0.000000,0.000000
4,chi.14718491,act,15.0,6.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,0.000000,0.000000,10499,0.000571,1.945910
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5282195,wu.89119131142,prouds,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282196,wu.89119131142,facilitator,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282197,wu.89119131142,teacher-librarian,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282198,wu.89119131142,pandani,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000


In [146]:
with open(Path("~/Projects/hypocognition/data/external/stopwords.txt").expanduser()) as f:
    words = [line.strip() for line in f if line.strip()]

In [147]:
# remove stop words
bila_nouns_full = bila_nouns_full[~bila_nouns_full["word"].isin(words)]
bila_nouns_full

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,ability,2.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-1.424169,10499,0.000190,1.098612
1,chi.14718491,accomplice,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.156552,10499,0.000000,0.000000
2,chi.14718491,account,14.0,9.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-7.520776,-2.250524,10499,0.000857,2.302585
3,chi.14718491,acre,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.444142,10499,0.000000,0.000000
6,chi.14718491,actor,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-1.633890,10499,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5282195,wu.89119131142,prouds,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282196,wu.89119131142,facilitator,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282197,wu.89119131142,teacher-librarian,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282198,wu.89119131142,pandani,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000


In [148]:
tot_words = bila_nouns_full.groupby('id')['word'].nunique().reset_index(name='Total_words_in_dict')
tot_counts = bila_nouns_full.groupby('id')['count'].sum().reset_index(name='Total_counts_in_dict')


In [149]:
# I think we will needs these later
dictionary_means = (
    bila_nouns_full
        .groupby('id', as_index=False)['count']
        .mean()
        .rename(columns={'count': 'dictionary_count_mean'})
)


In [150]:
# filter just the emotions
bila_nouns_full_emotions = bila_nouns_full[bila_nouns_full['word'].isin(emotions)].reset_index(drop=True)
bila_nouns_full_emotions

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,fear,8.0,9.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-7.520776,-1.337584,10499,0.000857,2.302585
1,chi.14718491,pleasure,5.0,5.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.031819,-1.571145,10499,0.000476,1.791759
2,chi.14718491,regret,5.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-0.486657,10499,0.000190,1.098612
3,chi.14718491,admiration,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.710507,10499,0.000000,0.000000
4,chi.14718491,anger,5.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-2.335062,10499,0.000190,1.098612
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11083,wu.89119131142,disgust,3.0,16.0,Herero,here1253,1989.0,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-7.795743,3.869782,33359,0.000480,2.833213
11084,wu.89119131142,frustration,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-10.629344,-0.596842,33359,0.000000,0.000000
11085,wu.89119131142,loneliness,3.0,4.0,Herero,here1253,1989.0,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.019809,1.825273,33359,0.000120,1.609438
11086,wu.89119131142,relief,11.0,10.0,Herero,here1253,1989.0,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-8.231207,0.866433,33359,0.000300,2.397895


the dataset is missing two of the survey emotions, {'calm', 'moved'}

In [151]:
filtered.columns

Index(['country', 'countryname', 'ISO3', 'admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness', 'Code',
       'Language_Name', 'lang_responses', 'total_responses', 'percent',
       'bila_language_name_mapping',
       'possible_alternative_bila_language_name_mappings', 'Selected'],
      dtype='object')

In [152]:
filtered['bila_language_name_mapping'].nunique()

1

In [153]:
filtered['Language_Name'].nunique()

1

In [154]:
filtered['countryname'].nunique()

3

In [155]:
# already filtered out everything but the 18 emotions
# now we want to filter evrything but the 29 dictionaries
# Where are lang_name and Language_Names the Same?
bila_nouns_full_emotions_filtered = bila_nouns_full_emotions[bila_nouns_full_emotions['langname'].isin(filtered['bila_language_name_mapping'].unique())].reset_index(drop=True)
bila_nouns_full_emotions_filtered

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,mdp.39015043036436,fear,8.0,57.0,Arabic,stan1318,1999.0,al-Mawrid : qāmūs ʻArabī-Inkilīzī / Rūḥī Baʻla...,"Dār al-ʻIlm lil-Malāyīn, 1999., دار العلم للمل...",0,Eurasia,Afro-Asiatic,"Afro-Asiatic, Semitic, West Semitic, Central S...",43.8525,27.9625,-7.235694,-1.050424,72677,0.000784,4.060443
1,mdp.39015043036436,pleasure,5.0,43.0,Arabic,stan1318,1999.0,al-Mawrid : qāmūs ʻArabī-Inkilīzī / Rūḥī Baʻla...,"Dār al-ʻIlm lil-Malāyīn, 1999., دار العلم للمل...",0,Eurasia,Afro-Asiatic,"Afro-Asiatic, Semitic, West Semitic, Central S...",43.8525,27.9625,-7.512121,-0.807742,72677,0.000592,3.784190
2,mdp.39015043036436,regret,5.0,15.0,Arabic,stan1318,1999.0,al-Mawrid : qāmūs ʻArabī-Inkilīzī / Rūḥī Baʻla...,"Dār al-ʻIlm lil-Malāyīn, 1999., دار العلم للمل...",0,Eurasia,Afro-Asiatic,"Afro-Asiatic, Semitic, West Semitic, Central S...",43.8525,27.9625,-8.524069,-0.319646,72677,0.000206,2.772589
3,mdp.39015043036436,admiration,3.0,4.0,Arabic,stan1318,1999.0,al-Mawrid : qāmūs ʻArabī-Inkilīzī / Rūḥī Baʻla...,"Dār al-ʻIlm lil-Malāyīn, 1999., دار العلم للمل...",0,Eurasia,Afro-Asiatic,"Afro-Asiatic, Semitic, West Semitic, Central S...",43.8525,27.9625,-9.687357,-1.283189,72677,0.000055,1.609438
4,mdp.39015043036436,anger,5.0,49.0,Arabic,stan1318,1999.0,al-Mawrid : qāmūs ʻArabī-Inkilīzī / Rūḥī Baʻla...,"Dār al-ʻIlm lil-Malāyīn, 1999., دار العلم للمل...",0,Eurasia,Afro-Asiatic,"Afro-Asiatic, Semitic, West Semitic, Central S...",43.8525,27.9625,-7.384213,-0.052223,72677,0.000674,3.912023
5,mdp.39015043036436,compassion,2.0,20.0,Arabic,stan1318,1999.0,al-Mawrid : qāmūs ʻArabī-Inkilīzī / Rūḥī Baʻla...,"Dār al-ʻIlm lil-Malāyīn, 1999., دار العلم للمل...",0,Eurasia,Afro-Asiatic,"Afro-Asiatic, Semitic, West Semitic, Central S...",43.8525,27.9625,-8.252074,2.512722,72677,0.000275,3.044522
6,mdp.39015043036436,love,10.0,134.0,Arabic,stan1318,1999.0,al-Mawrid : qāmūs ʻArabī-Inkilīzī / Rūḥī Baʻla...,"Dār al-ʻIlm lil-Malāyīn, 1999., دار العلم للمل...",0,Eurasia,Afro-Asiatic,"Afro-Asiatic, Semitic, West Semitic, Central S...",43.8525,27.9625,-6.389905,3.972935,72677,0.001844,4.905275
7,mdp.39015043036436,anxiety,2.0,22.0,Arabic,stan1318,1999.0,al-Mawrid : qāmūs ʻArabī-Inkilīzī / Rūḥī Baʻla...,"Dār al-ʻIlm lil-Malāyīn, 1999., دار العلم للمل...",0,Eurasia,Afro-Asiatic,"Afro-Asiatic, Semitic, West Semitic, Central S...",43.8525,27.9625,-8.161077,1.108333,72677,0.000303,3.135494
8,mdp.39015043036436,confusion,5.0,32.0,Arabic,stan1318,1999.0,al-Mawrid : qāmūs ʻArabī-Inkilīzī / Rūḥī Baʻla...,"Dār al-ʻIlm lil-Malāyīn, 1999., دار العلم للمل...",0,Eurasia,Afro-Asiatic,"Afro-Asiatic, Semitic, West Semitic, Central S...",43.8525,27.9625,-7.799940,1.064825,72677,0.000440,3.496508
9,mdp.39015043036436,determination,5.0,20.0,Arabic,stan1318,1999.0,al-Mawrid : qāmūs ʻArabī-Inkilīzī / Rūḥī Baʻla...,"Dār al-ʻIlm lil-Malāyīn, 1999., دار العلم للمل...",0,Eurasia,Afro-Asiatic,"Afro-Asiatic, Semitic, West Semitic, Central S...",43.8525,27.9625,-8.252074,3.539641,72677,0.000275,3.044522


In [156]:
print(bila_nouns_full_emotions_filtered['langname'].nunique())

1


In [157]:
filtered = filtered.loc[:, ~filtered.columns.duplicated()]

In [158]:

bila_nouns_full_emotions_filtered['langname'].nunique()

1

In [159]:
word_counts = (
    bila_nouns_full_emotions_filtered
    .groupby("langname")["word"]
    .nunique()
    .reset_index(name="n_unique_words")
)
word_counts

,langname,n_unique_words
0,Arabic,18


In [160]:
bila_nouns_full_emotions_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      18 non-null     object 
 1   word                    18 non-null     object 
 2   nsenses                 18 non-null     float64
 3   count                   18 non-null     float64
 4   langname                18 non-null     object 
 5   glottocode              18 non-null     object 
 6   year                    18 non-null     float64
 7   title                   18 non-null     object 
 8   imprint                 18 non-null     object 
 9   author                  18 non-null     object 
 10  area                    18 non-null     object 
 11  langfamily              18 non-null     object 
 12  affiliation             18 non-null     object 
 13  longitude               18 non-null     float64
 14  latitude                18 non-null     floa

In [161]:
# add the rows of the emotions words which don't appear in each dictionary

full_index = pd.MultiIndex.from_product(
    [bila_nouns_full_emotions_filtered["id"].unique(), bila_nouns_full_emotions_filtered["word"].unique()],
    names=["id", "word"]
)


In [162]:
# use the full index to add the missing values
bila_nouns_full_emotions_filtered_full = (
    bila_nouns_full_emotions_filtered
    .set_index(["id", "word"])
    .reindex(full_index)
    .reset_index()
)
num_cols = ["nsenses", "count", "log_count", 'regression_elaboration', 'dictsize_data', 'simple_elaboration']
bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      18 non-null     object 
 1   word                    18 non-null     object 
 2   nsenses                 18 non-null     float64
 3   count                   18 non-null     float64
 4   langname                18 non-null     object 
 5   glottocode              18 non-null     object 
 6   year                    18 non-null     float64
 7   title                   18 non-null     object 
 8   imprint                 18 non-null     object 
 9   author                  18 non-null     object 
 10  area                    18 non-null     object 
 11  langfamily              18 non-null     object 
 12  affiliation             18 non-null     object 
 13  longitude               18 non-null     float64
 14  latitude                18 non-null     floa

In [163]:

#set the number columns to 0 where NaN
bila_nouns_full_emotions_filtered_full[num_cols] = bila_nouns_full_emotions_filtered_full[num_cols].fillna(0)
meta_cols = [
    "langname", "glottocode", "year", "title", "imprint", "author",
    "area", "langfamily", "affiliation", "longitude", "latitude"
]
bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      18 non-null     object 
 1   word                    18 non-null     object 
 2   nsenses                 18 non-null     float64
 3   count                   18 non-null     float64
 4   langname                18 non-null     object 
 5   glottocode              18 non-null     object 
 6   year                    18 non-null     float64
 7   title                   18 non-null     object 
 8   imprint                 18 non-null     object 
 9   author                  18 non-null     object 
 10  area                    18 non-null     object 
 11  langfamily              18 non-null     object 
 12  affiliation             18 non-null     object 
 13  longitude               18 non-null     float64
 14  latitude                18 non-null     floa

In [164]:
# fill the those zero rows with dictionary meta -data
bila_nouns_full_emotions_filtered_full[meta_cols] = (
    bila_nouns_full_emotions_filtered_full
    .groupby("id")[meta_cols]
    .transform("first")
)

bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      18 non-null     object 
 1   word                    18 non-null     object 
 2   nsenses                 18 non-null     float64
 3   count                   18 non-null     float64
 4   langname                18 non-null     object 
 5   glottocode              18 non-null     object 
 6   year                    18 non-null     float64
 7   title                   18 non-null     object 
 8   imprint                 18 non-null     object 
 9   author                  18 non-null     object 
 10  area                    18 non-null     object 
 11  langfamily              18 non-null     object 
 12  affiliation             18 non-null     object 
 13  longitude               18 non-null     float64
 14  latitude                18 non-null     floa

In [165]:
# now we're don emaking datasets, this is a table of just the stuff we need for correlations
elab_per_emotion = bila_nouns_full_emotions_filtered_full[['langname',   'glottocode','word', 'count',  'id', 'year',"simple_elaboration", "regression_elaboration",	"dictsize_data"	]]
elab_per_emotion

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data
0,Arabic,stan1318,fear,57.0,mdp.39015043036436,1999.0,0.000784,-1.050424,72677
1,Arabic,stan1318,pleasure,43.0,mdp.39015043036436,1999.0,0.000592,-0.807742,72677
2,Arabic,stan1318,regret,15.0,mdp.39015043036436,1999.0,0.000206,-0.319646,72677
3,Arabic,stan1318,admiration,4.0,mdp.39015043036436,1999.0,0.000055,-1.283189,72677
4,Arabic,stan1318,anger,49.0,mdp.39015043036436,1999.0,0.000674,-0.052223,72677
5,Arabic,stan1318,compassion,20.0,mdp.39015043036436,1999.0,0.000275,2.512722,72677
6,Arabic,stan1318,love,134.0,mdp.39015043036436,1999.0,0.001844,3.972935,72677
7,Arabic,stan1318,anxiety,22.0,mdp.39015043036436,1999.0,0.000303,1.108333,72677
8,Arabic,stan1318,confusion,32.0,mdp.39015043036436,1999.0,0.000440,1.064825,72677
9,Arabic,stan1318,determination,20.0,mdp.39015043036436,1999.0,0.000275,3.539641,72677


In [166]:
elab_per_emotion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   langname                18 non-null     object 
 1   glottocode              18 non-null     object 
 2   word                    18 non-null     object 
 3   count                   18 non-null     float64
 4   id                      18 non-null     object 
 5   year                    18 non-null     float64
 6   simple_elaboration      18 non-null     float64
 7   regression_elaboration  18 non-null     float64
 8   dictsize_data           18 non-null     int64  
dtypes: float64(4), int64(1), object(4)
memory usage: 1.4+ KB


In [167]:
# Make table of the 18 languages for each country
stats_by_country = (
    filtered
    .groupby(['countryname', 'bila_language_name_mapping'])[emotions]
    .agg(['mean', 'std'])
)
stats_by_country = pd.DataFrame(stats_by_country)
stats_by_country

admiration                calm  \
                                             mean       std      mean   
countryname bila_language_name_mapping                                  
Germany     Arabic                       2.008403  1.980822  3.272727   
Netherlands Arabic                       2.419580  1.836303  3.545455   
Turkey      Arabic                       2.240000  1.814655  3.402597   

                                                 compassion            \
                                             std       mean       std   
countryname bila_language_name_mapping                                  
Germany     Arabic                      2.190890   3.950820  2.064467   
Netherlands Arabic                      1.689767   4.267606  1.848578   
Turkey      Arabic                      1.907463   4.012987  1.743360   

                                       determination               moved  \
                                                mean       std      mean   
countryname bila_language_name_mapping                                     
Germany     Arabic                          2.768595  1.918160  3.573770   
Netherlands Arabic                          3.580420  1.809259  3.654930   
Turkey      Arabic                          3.194805  1.849886  3.276316   

                                                  ...      fear            \
                                             std  ...      mean       std   
countryname bila_language_name_mapping            ...                       
Germany     Arabic                      1.978889  ...  2.278689  1.959348   
Netherlands Arabic                      1.883281  ...  2.692308  2.076803   
Turkey      Arabic                      1.970439  ...  2.220779  2.017185   

                                       frustration           loneliness  \
                                              mean       std       mean   
countryname bila_language_name_mapping                                    
Germany     Arabic                        2.639344  2.163046   2.804878   
Netherlands Arabic                        2.685315  1.980253   2.701389   
Turkey      Arabic                        2.763158  1.972264   3.026316   

                                                    regret            \
                                             std      mean       std   
countryname bila_language_name_mapping                                 
Germany     Arabic                      2.321173  1.909836  2.124256   
Netherlands Arabic                      2.225346  1.875000  1.946397   
Turkey      Arabic                      2.196808  2.027027  2.164299   

                                         sadness            
                                            mean       std  
countryname bila_language_name_mapping                      
Germany     Arabic                      3.227642  2.071718  
Netherlands Arabic                      3.270833  1.947632  
Turkey      Arabic                      2.987013  2.074237  

[3 rows x 40 columns]

In [168]:
# lets just flatten out the 3 level column names
stats_by_country.columns = [
    f"{emotion}_{stat}" for emotion, stat in stats_by_country.columns
]
stats_by_country = stats_by_country.reset_index()
stats_by_country

,countryname,bila_language_name_mapping,admiration_mean,admiration_std,calm_mean,calm_std,compassion_mean,compassion_std,determination_mean,determination_std,...,fear_mean,fear_std,frustration_mean,frustration_std,loneliness_mean,loneliness_std,regret_mean,regret_std,sadness_mean,sadness_std
0,Germany,Arabic,2.008403,1.980822,3.272727,2.190890,3.950820,2.064467,2.768595,1.918160,...,2.278689,1.959348,2.639344,2.163046,2.804878,2.321173,1.909836,2.124256,3.227642,2.071718
1,Netherlands,Arabic,2.419580,1.836303,3.545455,1.689767,4.267606,1.848578,3.580420,1.809259,...,2.692308,2.076803,2.685315,1.980253,2.701389,2.225346,1.875000,1.946397,3.270833,1.947632
2,Turkey,Arabic,2.240000,1.814655,3.402597,1.907463,4.012987,1.743360,3.194805,1.849886,...,2.220779,2.017185,2.763158,1.972264,3.026316,2.196808,2.027027,2.164299,2.987013,2.074237


In [169]:
# it was very wide, leyts make it narrow, longer, and more robust
stats_long = (
    stats_by_country
    .set_index(['countryname', 'bila_language_name_mapping'])
    .filter(regex='_(mean|std)$')
    .stack()
    .reset_index()
)

stats_long[['word', 'stat']] = stats_long['level_2'].str.rsplit('_', n=1, expand=True)
stats_long = stats_long.rename(columns={0: 'value'}).drop(columns='level_2')

stats_long = (
    stats_long
    .pivot_table(
        index=['countryname', 'bila_language_name_mapping', 'word'],
        columns='stat',
        values='value'
    )
    .reset_index()
)


In [170]:
#Merge our cleaned survey data with the BILA dictionary data
merged = elab_per_emotion.merge(
    stats_long,
    left_on=['langname', 'word'],
    right_on=['bila_language_name_mapping', 'word'],
    how='left'
)

merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,mean,std
0,Arabic,stan1318,fear,57.0,mdp.39015043036436,1999.0,0.000784,-1.050424,72677,Germany,Arabic,2.278689,1.959348
1,Arabic,stan1318,fear,57.0,mdp.39015043036436,1999.0,0.000784,-1.050424,72677,Netherlands,Arabic,2.692308,2.076803
2,Arabic,stan1318,fear,57.0,mdp.39015043036436,1999.0,0.000784,-1.050424,72677,Turkey,Arabic,2.220779,2.017185
3,Arabic,stan1318,pleasure,43.0,mdp.39015043036436,1999.0,0.000592,-0.807742,72677,Germany,Arabic,3.073770,2.117101
4,Arabic,stan1318,pleasure,43.0,mdp.39015043036436,1999.0,0.000592,-0.807742,72677,Netherlands,Arabic,3.370629,1.945302
5,Arabic,stan1318,pleasure,43.0,mdp.39015043036436,1999.0,0.000592,-0.807742,72677,Turkey,Arabic,3.197368,1.825886
6,Arabic,stan1318,regret,15.0,mdp.39015043036436,1999.0,0.000206,-0.319646,72677,Germany,Arabic,1.909836,2.124256
7,Arabic,stan1318,regret,15.0,mdp.39015043036436,1999.0,0.000206,-0.319646,72677,Netherlands,Arabic,1.875000,1.946397
8,Arabic,stan1318,regret,15.0,mdp.39015043036436,1999.0,0.000206,-0.319646,72677,Turkey,Arabic,2.027027,2.164299
9,Arabic,stan1318,admiration,4.0,mdp.39015043036436,1999.0,0.000055,-1.283189,72677,Germany,Arabic,2.008403,1.980822


In [171]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   langname                    54 non-null     object 
 1   glottocode                  54 non-null     object 
 2   word                        54 non-null     object 
 3   count                       54 non-null     float64
 4   id                          54 non-null     object 
 5   year                        54 non-null     float64
 6   simple_elaboration          54 non-null     float64
 7   regression_elaboration      54 non-null     float64
 8   dictsize_data               54 non-null     int64  
 9   countryname                 54 non-null     object 
 10  bila_language_name_mapping  54 non-null     object 
 11  mean                        54 non-null     float64
 12  std                         54 non-null     float64
dtypes: float64(6), int64(1), object(6)
me

In [ ]:
# add the dictionary_count_mean column
merged = merged.merge(
    dictionary_means,
    left_on='id',
    right_on='id',
    how='left'
)
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,mean,std,dictionary_count_mean
0,Arabic,stan1318,fear,57.0,mdp.39015043036436,1999.0,0.000784,-1.050424,72677,Germany,Arabic,2.278689,1.959348,8.662336
1,Arabic,stan1318,fear,57.0,mdp.39015043036436,1999.0,0.000784,-1.050424,72677,Netherlands,Arabic,2.692308,2.076803,8.662336
2,Arabic,stan1318,fear,57.0,mdp.39015043036436,1999.0,0.000784,-1.050424,72677,Turkey,Arabic,2.220779,2.017185,8.662336
3,Arabic,stan1318,pleasure,43.0,mdp.39015043036436,1999.0,0.000592,-0.807742,72677,Germany,Arabic,3.073770,2.117101,8.662336
4,Arabic,stan1318,pleasure,43.0,mdp.39015043036436,1999.0,0.000592,-0.807742,72677,Netherlands,Arabic,3.370629,1.945302,8.662336
5,Arabic,stan1318,pleasure,43.0,mdp.39015043036436,1999.0,0.000592,-0.807742,72677,Turkey,Arabic,3.197368,1.825886,8.662336
6,Arabic,stan1318,regret,15.0,mdp.39015043036436,1999.0,0.000206,-0.319646,72677,Germany,Arabic,1.909836,2.124256,8.662336
7,Arabic,stan1318,regret,15.0,mdp.39015043036436,1999.0,0.000206,-0.319646,72677,Netherlands,Arabic,1.875000,1.946397,8.662336
8,Arabic,stan1318,regret,15.0,mdp.39015043036436,1999.0,0.000206,-0.319646,72677,Turkey,Arabic,2.027027,2.164299,8.662336
9,Arabic,stan1318,admiration,4.0,mdp.39015043036436,1999.0,0.000055,-1.283189,72677,Germany,Arabic,2.008403,1.980822,8.662336


In [173]:
merged = merged[merged['countryname'].notna()]

In [175]:
# clarify the meaning of mean
merged = merged.rename(columns={'mean': "response_mean"})
# Clarify what std we're talking about
merged = merged.rename(columns={'std': "response_std"})
# drop a bunch of columns

merged = merged[['langname', 'glottocode', 'word', 'count', 'id', 'year',
       'simple_elaboration', 'regression_elaboration', 'dictsize_data',
       'countryname', 'bila_language_name_mapping', 'response_mean',
       'response_std', 'dictionary_count_mean' ]]
merged = merged.sort_values(by=["langname", 'word'])
merged


,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
9,Arabic,stan1318,admiration,4.0,mdp.39015043036436,1999.0,0.000055,-1.283189,72677,Germany,Arabic,2.008403,1.980822,8.662336
10,Arabic,stan1318,admiration,4.0,mdp.39015043036436,1999.0,0.000055,-1.283189,72677,Netherlands,Arabic,2.419580,1.836303,8.662336
11,Arabic,stan1318,admiration,4.0,mdp.39015043036436,1999.0,0.000055,-1.283189,72677,Turkey,Arabic,2.240000,1.814655,8.662336
12,Arabic,stan1318,anger,49.0,mdp.39015043036436,1999.0,0.000674,-0.052223,72677,Germany,Arabic,2.859504,1.831692,8.662336
13,Arabic,stan1318,anger,49.0,mdp.39015043036436,1999.0,0.000674,-0.052223,72677,Netherlands,Arabic,2.783217,1.835257,8.662336
14,Arabic,stan1318,anger,49.0,mdp.39015043036436,1999.0,0.000674,-0.052223,72677,Turkey,Arabic,2.826667,1.982173,8.662336
21,Arabic,stan1318,anxiety,22.0,mdp.39015043036436,1999.0,0.000303,1.108333,72677,Germany,Arabic,3.344262,1.817138,8.662336
22,Arabic,stan1318,anxiety,22.0,mdp.39015043036436,1999.0,0.000303,1.108333,72677,Netherlands,Arabic,3.440559,1.710046,8.662336
23,Arabic,stan1318,anxiety,22.0,mdp.39015043036436,1999.0,0.000303,1.108333,72677,Turkey,Arabic,3.500000,1.843909,8.662336
51,Arabic,stan1318,boredom,5.0,mdp.39015043036436,1999.0,0.000069,1.288125,72677,Germany,Arabic,3.421488,2.052446,8.662336


In [176]:
merged = merged.reset_index(drop=True)
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
0,Arabic,stan1318,admiration,4.0,mdp.39015043036436,1999.0,0.000055,-1.283189,72677,Germany,Arabic,2.008403,1.980822,8.662336
1,Arabic,stan1318,admiration,4.0,mdp.39015043036436,1999.0,0.000055,-1.283189,72677,Netherlands,Arabic,2.419580,1.836303,8.662336
2,Arabic,stan1318,admiration,4.0,mdp.39015043036436,1999.0,0.000055,-1.283189,72677,Turkey,Arabic,2.240000,1.814655,8.662336
3,Arabic,stan1318,anger,49.0,mdp.39015043036436,1999.0,0.000674,-0.052223,72677,Germany,Arabic,2.859504,1.831692,8.662336
4,Arabic,stan1318,anger,49.0,mdp.39015043036436,1999.0,0.000674,-0.052223,72677,Netherlands,Arabic,2.783217,1.835257,8.662336
5,Arabic,stan1318,anger,49.0,mdp.39015043036436,1999.0,0.000674,-0.052223,72677,Turkey,Arabic,2.826667,1.982173,8.662336
6,Arabic,stan1318,anxiety,22.0,mdp.39015043036436,1999.0,0.000303,1.108333,72677,Germany,Arabic,3.344262,1.817138,8.662336
7,Arabic,stan1318,anxiety,22.0,mdp.39015043036436,1999.0,0.000303,1.108333,72677,Netherlands,Arabic,3.440559,1.710046,8.662336
8,Arabic,stan1318,anxiety,22.0,mdp.39015043036436,1999.0,0.000303,1.108333,72677,Turkey,Arabic,3.500000,1.843909,8.662336
9,Arabic,stan1318,boredom,5.0,mdp.39015043036436,1999.0,0.000069,1.288125,72677,Germany,Arabic,3.421488,2.052446,8.662336


In [177]:
print(merged['langname'].nunique(),
      merged['bila_language_name_mapping'].nunique(),
merged['word'].nunique(),
merged['countryname'].nunique(), sep="\n")

1
1
18
3


In [178]:
merged = merged.sort_values(by=["langname", 'countryname','word'])
merged = merged.reset_index(drop=True)
merged = merged.fillna(0)


In [179]:
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
0,Arabic,stan1318,admiration,4.0,mdp.39015043036436,1999.0,0.000055,-1.283189,72677,Germany,Arabic,2.008403,1.980822,8.662336
1,Arabic,stan1318,anger,49.0,mdp.39015043036436,1999.0,0.000674,-0.052223,72677,Germany,Arabic,2.859504,1.831692,8.662336
2,Arabic,stan1318,anxiety,22.0,mdp.39015043036436,1999.0,0.000303,1.108333,72677,Germany,Arabic,3.344262,1.817138,8.662336
3,Arabic,stan1318,boredom,5.0,mdp.39015043036436,1999.0,0.000069,1.288125,72677,Germany,Arabic,3.421488,2.052446,8.662336
4,Arabic,stan1318,compassion,20.0,mdp.39015043036436,1999.0,0.000275,2.512722,72677,Germany,Arabic,3.950820,2.064467,8.662336
5,Arabic,stan1318,confusion,32.0,mdp.39015043036436,1999.0,0.000440,1.064825,72677,Germany,Arabic,2.433333,1.947778,8.662336
6,Arabic,stan1318,determination,20.0,mdp.39015043036436,1999.0,0.000275,3.539641,72677,Germany,Arabic,2.768595,1.918160,8.662336
7,Arabic,stan1318,disgust,28.0,mdp.39015043036436,1999.0,0.000385,4.333985,72677,Germany,Arabic,1.586777,1.686562,8.662336
8,Arabic,stan1318,fear,57.0,mdp.39015043036436,1999.0,0.000784,-1.050424,72677,Germany,Arabic,2.278689,1.959348,8.662336
9,Arabic,stan1318,frustration,16.0,mdp.39015043036436,1999.0,0.000220,6.448571,72677,Germany,Arabic,2.639344,2.163046,8.662336


In [180]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   langname                    54 non-null     object 
 1   glottocode                  54 non-null     object 
 2   word                        54 non-null     object 
 3   count                       54 non-null     float64
 4   id                          54 non-null     object 
 5   year                        54 non-null     float64
 6   simple_elaboration          54 non-null     float64
 7   regression_elaboration      54 non-null     float64
 8   dictsize_data               54 non-null     int64  
 9   countryname                 54 non-null     object 
 10  bila_language_name_mapping  54 non-null     object 
 11  response_mean               54 non-null     float64
 12  response_std                54 non-null     float64
 13  dictionary_count_mean       54 non-nu

In [182]:
merged.to_csv(Path("~/Projects/hypocognition/data/processed/covid_bila_merge.csv").expanduser())

"Moving forward, could you create a table with the following columns (broken down into the 36 samples):
* The sample country
* The language
* The correlation between the log count and the response means
* The correlation between the log count and the response SDs
* The correlation between the log count and the absolute distance of the response means from the scale midpoint

In [183]:
merged['response_mean'].mean()

np.float64(2.941825947484964)

In [184]:
import numpy as np

merged2 = merged.copy()

# log count
merged2["log_count"] = np.log(merged2["count"] + 1)

# absolute distance from midpoint
SCALE_MIDPOINT = merged2["response_mean"].mean()
merged2["abs_dist_midpoint"] = (
    merged2["response_mean"] - SCALE_MIDPOINT
).abs()

result = (
    merged2
    .groupby(["countryname", "langname", "id"])
    .agg(
        corr_logcount_response_mean=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "response_mean"]
            )
        ),
        corr_logcount_response_sd=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "response_std"]
            )
        ),
        corr_logcount_abs_dist_midpoint=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "abs_dist_midpoint"]
            )
        ),

    )
    .reset_index()
)


In [185]:
result

,countryname,langname,id,corr_logcount_response_mean,corr_logcount_response_sd,corr_logcount_abs_dist_midpoint
0,Germany,Arabic,mdp.39015043036436,-0.144513,-0.294592,-0.127128
1,Netherlands,Arabic,mdp.39015043036436,-0.050352,-0.066060,-0.181116
2,Turkey,Arabic,mdp.39015043036436,-0.069355,-0.224864,0.032368


In [186]:
result.to_csv(Path("~/Projects/hypocognition/data/processed/ARABIC2corr_covid_bila.csv").expanduser())